# Module 20 — Caching & Profiling: Interactive Verification

## What you will discover

Every cell runs the module's **real** `cache_engine` and asserts its behaviour.

1. Why a cache miss returns a `MISSING` sentinel rather than `None` — and the
   bug that choice prevents.
2. That LRU eviction and TTL expiry are two different mechanisms with two
   different observable effects.
3. That single-flight collapses a stampede of concurrent misses into **one**
   load, measured in wall-clock time rather than asserted.
4. That the cache's own statistics agree with what actually happened.

**One cell near the end is deliberately broken.** Fixing it is the exercise.

## Setup

In [ ]:
import sys
import threading
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "project_solution"))

import cache_engine as ce

print(f"loaded from: {Path(ce.__file__).parent.name}/")
print(f"module sentinels: {[n for n in dir(ce) if n.isupper()]}")
print(f"TwoTierCache API: {[m for m in dir(ce.TwoTierCache) if not m.startswith('_')]}")

## 1. Why a miss is `MISSING`, not `None`

If a miss returned `None`, then a cached value that genuinely *is* `None`
becomes indistinguishable from an absent key — so every read of it re-runs the
expensive loader, forever, and the cache silently stops working for exactly the
keys whose answer is "nothing".

This is the kind of bug that never raises and never shows up in a hit-ratio
dashboard as anything but "that key is unpopular".

In [ ]:
cache = ce.LRUCache(capacity=8, default_ttl=60)

print(f"absent key    -> {cache.get('never-set')!r}")
assert cache.get("never-set") is ce.MISSING

cache.set("explicitly-none", None)
print(f"stored None   -> {cache.get('explicitly-none')!r}")

# The two are distinguishable, which is the entire point.
assert cache.get("explicitly-none") is None
assert cache.get("explicitly-none") is not ce.MISSING
assert cache.get("never-set") is not None

print("\nA stored None and an absent key are distinguishable.")
print("With a None-as-miss design, 'explicitly-none' would reload on every read.")

## 2. Predict before you run

A cache with `capacity=3` receives four distinct keys, in order `k0 k1 k2 k3`,
with nothing read in between.

Separately, a cache with `default_ttl=0.05` stores one key, then waits 120 ms.

**Write down your answers before running:**

1. After the four writes, what does `get("k0")` return? What about `get("k3")`?
2. After the wait, what does the second cache return for its key?
3. Both answers look the same when printed. What distinguishes an **eviction**
   from an **expiry**, and where would you see the difference?

Question 3 is the one worth thinking about.

In [ ]:
# --- eviction: capacity exceeded ---
evicting = ce.LRUCache(capacity=3, default_ttl=60)
for i in range(4):
    evicting.set(f"k{i}", i)

print(f"k0 (oldest, evicted): {evicting.get('k0')!r}")
print(f"k3 (newest, present): {evicting.get('k3')!r}")
assert evicting.get("k0") is ce.MISSING, "the least recently used key is evicted"
assert evicting.get("k3") == 3, "the newest key survives"

# --- expiry: TTL elapsed, capacity irrelevant ---
expiring = ce.LRUCache(capacity=64, default_ttl=0.05)
expiring.set("x", "value")
assert expiring.get("x") == "value", "present before the TTL elapses"
time.sleep(0.12)
print(f"\nx after 120ms with a 50ms TTL: {expiring.get('x')!r}")
assert expiring.get("x") is ce.MISSING

print("\nBoth return MISSING - but for different reasons, and the cache's own")
print("counters are where the difference is visible:\n")

ev_stats, ex_stats = evicting.stats.as_dict(), expiring.stats.as_dict()
print(f"  evicting: evictions={ev_stats['evictions']} expirations={ev_stats['expirations']}")
print(f"  expiring: evictions={ex_stats['evictions']} expirations={ex_stats['expirations']}")

assert ev_stats["evictions"] == 1, "capacity pressure records an eviction"
assert ev_stats["expirations"] == 0, "nothing expired - the TTL was 60s"
assert ex_stats["expirations"] == 1, "an elapsed TTL records an expiration"
assert ex_stats["evictions"] == 0, "nothing was evicted - capacity was 64"

print("\nThis is the operational difference: rising evictions means the cache")
print("is too small; rising expirations means the TTL is too short. The two")
print("call for opposite fixes, and MISSING alone cannot tell you which.")

## 3. Measurement — single-flight under a real stampede

The README claims single-flight collapses concurrent misses into one load. That
is a claim about wall-clock behaviour under threads, so it has to be *measured*,
not asserted from the code's shape.

40 threads ask for the same cold key at once. The loader sleeps 50 ms. If every
thread ran its own load, the work would total ~2 seconds.

In [ ]:
cache = ce.TwoTierCache(l1_capacity=16, l1_ttl=30, l2_ttl=300, l2=ce.InMemoryL2Backend())

loads = {"n": 0}
LOAD_MS = 0.05

def slow_loader():
    loads["n"] += 1
    time.sleep(LOAD_MS)
    return "expensive-value"

results = []
lock = threading.Lock()

def worker():
    v = cache.get_or_load("hot-key", slow_loader)
    with lock:
        results.append(v)

threads = [threading.Thread(target=worker) for _ in range(40)]
started = time.perf_counter()
for t in threads:
    t.start()
for t in threads:
    t.join()
elapsed = time.perf_counter() - started

print(f"threads:          {len(threads)}")
print(f"loader calls:     {loads['n']}")
print(f"wall clock:       {elapsed * 1000:.1f} ms")
print(f"serial would be:  {40 * LOAD_MS * 1000:.0f} ms")
print(f"all agree:        {len(set(results)) == 1}")

assert loads["n"] == 1, f"single-flight failed: {loads['n']} loads instead of 1"
assert len(set(results)) == 1, "every caller must receive the same value"
assert len(results) == 40, "no caller may be dropped"

# The wall-clock assertion is the one that would catch a lock that serialises
# the callers instead of collapsing them. Generous bound - this is a laptop.
assert elapsed < 40 * LOAD_MS / 4, (
    f"{elapsed * 1000:.0f} ms suggests callers serialised rather than collapsed"
)

report = cache.report()
print(f"\nstampedes_prevented: {report['stampedes_prevented']}")
assert report["stampedes_prevented"] == 39, "39 callers rode the one in-flight load"
assert report["loads"] == 1

## 4. The cache's own statistics must agree with reality

A hit-ratio dashboard that disagrees with what happened is worse than no
dashboard. Here the counters are checked against an exactly-known access
pattern.

In [ ]:
# An explicit in-memory L2, so this cell depends on no external server.
cache = ce.TwoTierCache(l1_capacity=4, l1_ttl=30, l2_ttl=300, l2=ce.InMemoryL2Backend())

cache.set("a", 1)
cache.set("b", 2)

hits = [cache.get("a"), cache.get("a"), cache.get("b")]   # 3 hits
misses = [cache.get("zz"), cache.get("yy")]               # 2 misses

report = cache.report()
for k in sorted(report):
    print(f"  {k:<22} {report[k]}")

assert hits == [1, 1, 2], f"hit values wrong: {hits}"
assert all(m is ce.MISSING for m in misses), "misses must be the sentinel"

# There is no flat `hits` key, and that is a design choice rather than an
# omission: "we are getting hits" and "we are getting them from the FAST tier"
# are different operational facts, so the tiers are counted separately.
total_hits = report["l1_hits"] + report["l2_hits"]
assert total_hits == 3, f"expected 3 hits, got {total_hits}"
assert report["misses"] == 2, f"expected 2 misses, got {report['misses']}"

# hit_ratio is rounded for display, so compare with a tolerance rather than ==.
assert abs(report["hit_ratio"] - 0.6) < 1e-4, f"ratio {report['hit_ratio']}"
assert report["l1_ratio"] == 1.0, "every hit here was served from L1"

print("\nThe counters agree with the exact access pattern above.")

## 5. Fix this cell

The expected values below are **wrong on purpose**. Run it, read the failure,
and correct them from what the cells above established.

Change only the expected values.

In [ ]:
# DELIBERATELY BROKEN - three expected values, two of them wrong. Fix in place.

expected_loads               = 40     # loader calls when 40 threads race one cold key
expected_stampedes_prevented = 39     # callers that rode the in-flight load
expected_hit_ratio           = 0.5    # after 3 hits and 2 misses

c = ce.TwoTierCache(l1_capacity=8, l1_ttl=30, l2_ttl=300, l2=ce.InMemoryL2Backend())
n = {"calls": 0}
def loader():
    n["calls"] += 1
    time.sleep(0.03)
    return "V"

ts = [threading.Thread(target=lambda: c.get_or_load("k", loader)) for _ in range(40)]
for t in ts:
    t.start()
for t in ts:
    t.join()

c2 = ce.TwoTierCache(l1_capacity=8, l2=ce.InMemoryL2Backend())
c2.set("a", 1)
for _ in range(3):
    c2.get("a")
c2.get("miss-1")
c2.get("miss-2")

assert expected_loads == n["calls"], f"loads: expected {expected_loads}, got {n['calls']}"
assert expected_stampedes_prevented == c.report()["stampedes_prevented"], \
    f"stampedes: got {c.report()['stampedes_prevented']}"
assert abs(expected_hit_ratio - c2.report()["hit_ratio"]) < 1e-9, \
    f"hit_ratio: got {c2.report()['hit_ratio']}"

print("All three match. Now explain WHY each number is what it is.")

## Takeaways

1. **A miss must be a sentinel, not `None`.** Otherwise a legitimately cached
   `None` reloads on every read, silently and forever, and nothing in a
   hit-ratio dashboard will tell you.
2. **Eviction and expiry are different mechanisms.** Both surface as `MISSING`;
   only the counters distinguish capacity pressure from staleness, and they
   call for opposite fixes.
3. **Single-flight must be measured in wall-clock time.** Asserting only that
   the loader ran once would pass for a lock that *serialises* forty callers —
   correct, and forty times too slow.
4. **Every caller must get the value, and the same value.** A stampede
   protection that drops callers or hands back different objects has traded one
   bug for a worse one.
5. **The counters must agree with an exactly-known access pattern.** If they
   do not, every capacity decision made from them is guesswork.

### Where to go next

- [`01_README.md`](01_README.md) — the concepts in depth
- [`project_solution/test_cache_engine.py`](project_solution/test_cache_engine.py) — the full test suite
- `debug_lab/` — planted defects to diagnose
- `starter/` — build it yourself; the shipped tests are the specification